# Activation Inspection: stacked PhasorDense on sfmnist

The puzzle: across multiple training runs of the sfmnist model,
**loss decreases steadily but test_acc bounces wildly between epochs**
(0.04 -> 0.29 -> 0.07 -> 0.18 ...), and the final epoch frequently
collapses below earlier-epoch accuracy. This pattern is consistent
across `n_layers in {1, 3}` and with/without residuals. Something is
moving epoch-over-epoch that doesn't show up in the training loss.

This notebook captures **per-layer activations on a fixed test batch**
both before training begins and after each of N epochs, so we can
visualize what the model is doing layer-by-layer and when it goes off
the rails.

What to look for:
- **Magnitude saturation**: `|z|` ~ 1 everywhere => `normalize_to_unit_circle`
  has flattened the signal; downstream layers see only phase, no
  amplitude information.
- **Phase collapse**: per-layer phase histograms cluster around a
  single value => the layer's output is not differentiating across
  inputs.
- **Per-channel activity**: dead channels show as flat rows in the
  per-channel std plot.
- **Across-epoch drift**: same fixed batch's activations should evolve
  smoothly; abrupt jumps in layer-N activations (with layers <N stable)
  suggest one layer's parameters are unstable.

## 1. Setup

In [ ]:
using PhasorNetworks
using Lux, Random, Zygote, Optimisers, Statistics
using CUDA, LuxCUDA
using MLDatasets: FashionMNIST
using OneHotArrays: onehotbatch
using Plots
using Random: Xoshiro
using Statistics: mean, std

# Pull build_sfmnist_model + helpers from scripts/sfmnist.jl
include(joinpath(@__DIR__, "..", "scripts", "sfmnist.jl"))

In [ ]:
# Knobs - tweak and re-run cells 3+ to compare configs.
const D            = 96
const B_INSPECT    = 64
const B_TRAIN      = 64
const N_TRAIN      = 5_000
const N_TEST       = 1_000
const N_EPOCHS     = 6
const LR           = 1f-3
const N_LAYERS     = 3
const USE_RESIDUAL = true
const ENCODING     = :phase
const MODE         = :hippo

USE_CUDA = CUDA.functional()
gdev = USE_CUDA ? gpu_device() : cpu_device()
@info "device" string(gdev)

## 2. Data + fixed inspection batch

In [ ]:
(Xtr, ytr), (Xte, yte) = load_fashionmnist(; n_train = N_TRAIN, n_test = N_TEST)
iter = ENCODING === :zoh ? batch_iterator : batch_iterator_phase
train_batches = iter(Xtr, ytr, B_TRAIN)
test_batches  = iter(Xte, yte, B_TRAIN)

# Fixed inspection batch: first B_INSPECT test samples. Same one used at every epoch.
inspect_X = ENCODING === :zoh ?
    pixel_to_phasor_batch(Xte[:, :, 1:B_INSPECT]) :
    pixel_to_phase_batch(Xte[:, :, 1:B_INSPECT])
inspect_y = yte[1:B_INSPECT] .+ 1
inspect_X_dev = inspect_X |> gdev
@info "data" size(inspect_X) eltype(inspect_X)

## 3. Model + instrumentation

`capture_layer_outputs` walks the Chain manually, calling each layer in
turn and saving each layer's output as a host array. Layers up to
the readout return Phase 3D `(C, L, B)`; the final `LastStepDense`
returns logits `(n_classes, B)`.

In [ ]:
rng = Xoshiro(0)
model = build_sfmnist_model(MODE, D;
                             n_layers = N_LAYERS,
                             use_residual = USE_RESIDUAL,
                             encoding = ENCODING)
ps_cpu, st_cpu = Lux.setup(rng, model)
ps = ps_cpu |> gdev
st = st_cpu |> gdev
opt_state = Optimisers.setup(Optimisers.Adam(Float32(LR)), ps)
@info "model" length(model.layers) typeof.(model.layers)

In [ ]:
"""
    capture_layer_outputs(model, ps, st, x) -> Vector

Walk Chain layer-by-layer; return each intermediate output as a host
array (one entry per layer).
"""
function capture_layer_outputs(model, ps, st, x)
    out_each = []
    cur = x
    for (i, layer) in enumerate(model.layers)
        ps_i = ps[Symbol("layer_$i")]
        st_i = st[Symbol("layer_$i")]
        cur, _ = layer(cur, ps_i, st_i)
        push!(out_each, (idx = i,
                         kind = string(typeof(layer).name.name),
                         out  = Array(cur)))
    end
    return out_each
end

# Sanity check: capture pre-training activations
pre_acts = capture_layer_outputs(model, ps, st, inspect_X_dev)
for a in pre_acts
    println("  layer $(a.idx) [$(a.kind)]: size = $(size(a.out)), eltype = $(eltype(a.out))")
end

## 4. Per-layer summary statistics

For each Phase-typed layer output `(C, L, B)`:
- **`|z|` distribution**: re-lift to complex via `angle_to_complex`,
  take magnitude. If always ~1, `normalize_to_unit_circle` is
  saturating.
- **Phase distribution**: histogram of Phase values per layer.
- **Per-channel activity**: std across timesteps x batch per channel.
  Dead channels have std ~ 0.

In [ ]:
function layer_stats(act::AbstractArray{<:Phase, 3})
    a_f  = Float32.(act)
    z    = Complex{Float32}.(angle_to_complex(act))
    mags = abs.(z)
    return (
        n_channels    = size(act, 1),
        L             = size(act, 2),
        B             = size(act, 3),
        phase_min     = minimum(a_f),
        phase_max     = maximum(a_f),
        phase_mean    = mean(a_f),
        phase_std     = std(a_f),
        mag_mean      = mean(mags),
        mag_std       = std(mags),
        per_chan_std  = std(a_f; dims = (2, 3)) |> vec,
        per_step_std  = std(a_f; dims = (1, 3)) |> vec,
        per_batch_std = std(a_f; dims = (1, 2)) |> vec,
    )
end

function logits_stats(logits::AbstractMatrix{<:Real}, labels::Vector{Int})
    preds = vec(getindex.(argmax(logits; dims = 1), 1))
    acc = mean(preds .== labels)
    return (acc = acc, mean = mean(logits), std = std(logits),
            per_class_mean = [mean(logits[c, :]) for c in 1:size(logits, 1)])
end

println("=== PRE-TRAINING ===")
for a in pre_acts
    if eltype(a.out) <: Phase
        s = layer_stats(a.out)
        n_dead = count(<(0.01), s.per_chan_std)
        println("  L$(a.idx) [$(a.kind)]: phase mu=$(round(s.phase_mean; digits=3)) sigma=$(round(s.phase_std; digits=3)) | mag mu=$(round(s.mag_mean; digits=3)) sigma=$(round(s.mag_std; digits=3)) | dead chans=$n_dead/$(s.n_channels)")
    else
        s = logits_stats(a.out, inspect_y)
        println("  L$(a.idx) [$(a.kind)]: acc=$(round(s.acc; digits=3)) | logit mu=$(round(s.mean; digits=3)) sigma=$(round(s.std; digits=3))")
    end
end

## 5. Train and snapshot per epoch

In [ ]:
snapshots = [pre_acts]   # epoch 0 = pre-training
losses_per_epoch = Float32[]
test_accs        = Float32[]

for epoch in 1:N_EPOCHS
    losses = Float32[]
    for (x, y) in train_batches
        x_d  = x |> gdev
        y_oh = Float32.(onehotbatch(y, 1:10)) |> gdev
        loss_val, back = Zygote.pullback(ps) do p
            logits, _ = model(x_d, p, st)
            ce_loss(logits, y_oh)
        end
        grads = back(one(loss_val))[1]
        opt_state, ps = Optimisers.update(opt_state, ps, grads)
        push!(losses, loss_val)
    end
    test_acc = evaluate(model, ps, st, test_batches; gdev)
    push!(losses_per_epoch, mean(losses))
    push!(test_accs, test_acc)
    push!(snapshots, capture_layer_outputs(model, ps, st, inspect_X_dev))
    @info "epoch" epoch loss = round(mean(losses); digits = 3) test_acc = round(test_acc; digits = 3)
end

## 6. Trajectory: loss + test_acc

In [ ]:
plot(1:N_EPOCHS, test_accs;
     label = "test_acc", marker = :circle, lw = 2, ylim = (0, 0.5),
     legend = :topleft, xlabel = "epoch", ylabel = "test accuracy",
     title = "Training trajectory ($N_LAYERS layers, residual=$USE_RESIDUAL)")
hline!([0.10], label = "chance", ls = :dash, color = :gray)

## 7. Per-layer phase distribution: pre vs post training

In [ ]:
phase_layers_idx = [a.idx for a in pre_acts if eltype(a.out) <: Phase]
final_idx        = length(snapshots)

plots = []
for li in phase_layers_idx
    pre  = Float32.(snapshots[1][li].out) |> vec
    post = Float32.(snapshots[final_idx][li].out) |> vec
    p = histogram(pre; bins = -1.0:0.05:1.0, label = "pre",
                  fillalpha = 0.5, color = :gray,
                  xlabel = "Phase", ylabel = "count",
                  title = "L$li [$(snapshots[1][li].kind)]")
    histogram!(p, post; bins = -1.0:0.05:1.0, label = "post (ep $N_EPOCHS)",
               fillalpha = 0.5, color = :red)
    push!(plots, p)
end
plot(plots...; layout = (length(plots), 1), size = (700, 220 * length(plots)))

## 8. Per-layer magnitude distribution

If `mag` collapses to ~1 throughout, `normalize_to_unit_circle` has
saturated the layer (only phase information survives downstream).

In [ ]:
plots_m = []
for li in phase_layers_idx
    pre_z  = angle_to_complex(snapshots[1][li].out)
    post_z = angle_to_complex(snapshots[final_idx][li].out)
    pre_m  = vec(abs.(pre_z))
    post_m = vec(abs.(post_z))
    p = histogram(pre_m; bins = 0.0:0.025:1.05, label = "pre",
                  fillalpha = 0.5, color = :gray,
                  xlabel = "|z|", ylabel = "count",
                  title = "L$li [$(snapshots[1][li].kind)] magnitude")
    histogram!(p, post_m; bins = 0.0:0.025:1.05, label = "post",
               fillalpha = 0.5, color = :red)
    push!(plots_m, p)
end
plot(plots_m...; layout = (length(plots_m), 1), size = (700, 220 * length(plots_m)))

## 9. Per-channel activity: how many channels are 'live'?

Dead channels have near-zero variance across batch x timesteps. If a
layer has many dead channels, it's bottlenecking the representation.

In [ ]:
plots_c = []
for li in phase_layers_idx
    pre_std  = vec(std(Float32.(snapshots[1][li].out); dims = (2, 3)))
    post_std = vec(std(Float32.(snapshots[final_idx][li].out); dims = (2, 3)))
    p = plot(sort(pre_std,  rev = true); label = "pre",  color = :gray, lw = 2,
             xlabel = "channel rank", ylabel = "std(phase across L*B)",
             title = "L$li [$(snapshots[1][li].kind)] per-channel activity")
    plot!(p, sort(post_std, rev = true); label = "post", color = :red, lw = 2)
    push!(plots_c, p)
end
plot(plots_c...; layout = (length(plots_c), 1), size = (700, 220 * length(plots_c)))

## 10. Activation drift across epochs

For the deepest pre-readout layer, trace the last-timestep phase of a
few channels x samples across epochs. Smooth = healthy training;
abrupt jumps = layer instability.

In [ ]:
li = phase_layers_idx[end]
sample_chans   = 1:min(8, size(snapshots[1][li].out, 1))
sample_batches = 1:min(8, B_INSPECT)

trace = zeros(Float32, length(snapshots), length(sample_chans), length(sample_batches))
for (e, snap) in enumerate(snapshots)
    a = Float32.(snap[li].out)
    L_end = size(a, 2)
    for (ci, c) in enumerate(sample_chans), (bi, b) in enumerate(sample_batches)
        trace[e, ci, bi] = a[c, L_end, b]
    end
end
plot()
for ci in 1:length(sample_chans), bi in 1:length(sample_batches)
    plot!(0:N_EPOCHS, trace[:, ci, bi]; label = "", lw = 0.7, alpha = 0.6)
end
plot!(xlabel = "epoch", ylabel = "Phase",
      title = "L$li last-step phase drift (8 chans x 8 samples)",
      ylim = (-1, 1))

## 11. Per-epoch readout accuracy on the inspection batch

Distinct from the test-set accuracy logged during training - this uses
the SAME fixed batch each epoch, so changes reflect model evolution
not data variance.

In [ ]:
inspect_acc = Float32[]
for snap in snapshots
    last_layer = snap[end]
    @assert !(eltype(last_layer.out) <: Phase) "last layer should be readout (logits)"
    s = logits_stats(last_layer.out, inspect_y)
    push!(inspect_acc, s.acc)
end
plot(0:N_EPOCHS, inspect_acc;
     marker = :circle, lw = 2, label = "inspection batch acc",
     xlabel = "epoch", ylabel = "accuracy", ylim = (0, 0.5),
     title = "Acc on fixed inspection batch")
hline!([0.10], label = "chance", ls = :dash, color = :gray)

## 12. Final-step readout: per-class logit means

If one class consistently gets the highest mean logit, the model has
collapsed to predict-everything-as-one-class. Would explain
sub-chance accuracy in some epochs.

In [ ]:
logit_class_means = zeros(Float32, length(snapshots), 10)
for (e, snap) in enumerate(snapshots)
    logits = snap[end].out
    logit_class_means[e, :] .= vec(mean(logits; dims = 2))
end
heatmap(logit_class_means;
        xlabel = "class", ylabel = "epoch (0=pre)",
        title = "Per-class logit mean over epochs",
        colorbar_title = "mean logit",
        c = :viridis, yflip = true)

## Notes on what to look at

If the trajectory in section 6 is bouncy and section 10 (drift) shows
abrupt jumps in the deeper layer's activations, the deeper layer is
overshooting each epoch -- either lr too high or lambda updates are
amplifying through depth. Check section 8 (magnitude) to see if
`normalize_to_unit_circle` has flattened the signal entirely; if mag
distribution after training is a spike at 1.0, the layer is using only
phase, no amplitude.

If section 12 (per-class logit heatmap) shows one column dominant at
the bad epochs, the model is collapsing to a single-class prediction --
likely the readout has learned a dead bias and the SSM activations are
irrelevant to it at that snapshot.